In [2]:
import os
import pandas as pd
import numpy as np
import pickle as pkl
from rdkit.Chem import MolFromSmiles, MolToInchiKey
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from rdkit.Chem.Descriptors import ExactMolWt
from pathlib import Path
#import mist.utils as utils
from pathlib import Path

ENABLE_FILE_EXPORTS = True

In [3]:
seed = 42
num_mols = 'all'

In [4]:
def write_mgf(df, output_path):
    if ENABLE_FILE_EXPORTS:
        with open(output_path, "w") as f:
            for idx, row in df.iterrows():
                mz_int_pairs = row['spec']
                f.write("BEGIN IONS\n")
                f.write(f"FEATURE_ID={row['compound']}\n")
                f.write(f"PEPMASS={row['parentmass']:.6f}\n")
                f.write("CHARGE=1+\n")
                f.write("MSLEVEL=1\n")
                f.write("IONIZATION_MODE=EI\n")
                f.write(f"TITLE={row['compound']}\n")
                for mz, intensity in mz_int_pairs:
                    f.write(f"{mz} {intensity}\n")
                f.write("END IONS\n\n")
    else:
        print(f'ENABLE_FILE_EXPORTS: {ENABLE_FILE_EXPORTS}')

## NEIMS gecko

In [4]:
path_to_data = f'../../../data/neims/gecko_EIMS_spectra'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)
fname = f"df_neims_gecko_TMS_post_3_9_22.pkl"
with open(fname, 'rb') as file:
    df_gecko = pkl.load(file)
df_gecko

,SMILES,spec
3,C[Si](C)(C)OC(=O)C(=O)[N+](=O)[O-],"[[14, 45], [15, 86], [16, 6], [18, 13], [25, 2..."
4,C[Si](C)(C)OOC(=O)C(=O)[N+](=O)[O-],"[[14, 67], [15, 193], [16, 20], [25, 27], [26,..."
5,C[Si](C)(C)OC(=O)C(=O)C(=O)[N+](=O)[O-],"[[14, 39], [15, 78], [16, 12], [18, 7], [25, 1..."
6,C[Si](C)(C)OOC(=O)C(=O)C(=O)[N+](=O)[O-],"[[14, 56], [15, 144], [16, 18], [18, 0], [25, ..."
7,C[Si](C)(C)OC(=O)C(O[Si](C)(C)C)C(=O)C(=O)C(=O...,"[[14, 31], [15, 47], [18, 18], [25, 5], [26, 4..."
...,...,...
166153,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,"[[14, 5], [15, 17], [16, 0], [26, 3], [27, 49]..."
166154,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,"[[2, 3], [8, 0], [14, 15], [15, 30], [18, 21],..."
166155,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,"[[14, 18], [15, 44], [16, 1], [18, 1], [20, 5]..."
166156,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,"[[14, 0], [15, 27], [26, 21], [27, 30], [28, 6..."


In [5]:
labels_df = pd.read_csv(f'{path_to_data}/mist_inputs/labels_tms.tsv', sep='\t')
labels_df['compound'] = labels_df['spec']
labels_df = labels_df.drop(columns=['Unnamed: 0', 'spec'])
labels_df

,dataset,ionization,formula,smiles,inchikey,instrument,compound
0,neims_tms,[M]+,C5H9NO5Si,C[Si](C)(C)OC(=O)C(=O)[N+](=O)[O-],CCWMGOZIQRPUPI-UHFFFAOYSA-N,simulated,gecko_tms_3
1,neims_tms,[M]+,C5H9NO6Si,C[Si](C)(C)OOC(=O)C(=O)[N+](=O)[O-],NLWVWMNHWHOVDG-UHFFFAOYSA-N,simulated,gecko_tms_4
2,neims_tms,[M]+,C6H9NO6Si,C[Si](C)(C)OC(=O)C(=O)C(=O)[N+](=O)[O-],KTPQPVBPMUNXAR-UHFFFAOYSA-N,simulated,gecko_tms_5
3,neims_tms,[M]+,C6H9NO7Si,C[Si](C)(C)OOC(=O)C(=O)C(=O)[N+](=O)[O-],FPCOVUBGSLBMMO-UHFFFAOYSA-N,simulated,gecko_tms_6
4,neims_tms,[M]+,C12H19NO9Si2,C[Si](C)(C)OC(=O)C(O[Si](C)(C)C)C(=O)C(=O)C(=O...,ZARPPVOVFLKXIU-UHFFFAOYSA-N,simulated,gecko_tms_7
...,...,...,...,...,...,...,...
159598,neims_tms,[M]+,C16H32N2O15Si3,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,JHFUNAOALBCUEU-UHFFFAOYSA-N,simulated,gecko_tms_166429
159599,neims_tms,[M]+,C16H33NO11Si3,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,WIPWTSILCDGNDN-UHFFFAOYSA-N,simulated,gecko_tms_166430
159600,neims_tms,[M]+,C13H23NO11Si2,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,TZYJPJSYLIRUID-UHFFFAOYSA-N,simulated,gecko_tms_166431
159601,neims_tms,[M]+,C12H22N2O14Si2,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,FAZHSBRCQRQGLZ-UHFFFAOYSA-N,simulated,gecko_tms_166432


In [6]:
df = pd.merge(df, labels_df, left_on='SMILES', right_on='smiles')
df

NameError: name 'df' is not defined

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['ionization'] = '[M]+'

In [ ]:
df

,SMILES,spec,dataset,ionization,formula,smiles,inchikey,instrument,compound,mols,parentmass
0,C[Si](C)(C)OC(=O)C(=O)[N+](=O)[O-],"[[14, 45], [15, 86], [16, 6], [18, 13], [25, 2...",neims_tms,[M]+,C5H9NO5Si,C[Si](C)(C)OC(=O)C(=O)[N+](=O)[O-],CCWMGOZIQRPUPI-UHFFFAOYSA-N,simulated,gecko_tms_3,<rdkit.Chem.rdchem.Mol object at 0x7fba2c82f0b0>,191.024999
1,C[Si](C)(C)OOC(=O)C(=O)[N+](=O)[O-],"[[14, 67], [15, 193], [16, 20], [25, 27], [26,...",neims_tms,[M]+,C5H9NO6Si,C[Si](C)(C)OOC(=O)C(=O)[N+](=O)[O-],NLWVWMNHWHOVDG-UHFFFAOYSA-N,simulated,gecko_tms_4,<rdkit.Chem.rdchem.Mol object at 0x7fba2c82f970>,207.019914
2,C[Si](C)(C)OC(=O)C(=O)C(=O)[N+](=O)[O-],"[[14, 39], [15, 78], [16, 12], [18, 7], [25, 1...",neims_tms,[M]+,C6H9NO6Si,C[Si](C)(C)OC(=O)C(=O)C(=O)[N+](=O)[O-],KTPQPVBPMUNXAR-UHFFFAOYSA-N,simulated,gecko_tms_5,<rdkit.Chem.rdchem.Mol object at 0x7fba2c82f9e0>,219.019914
3,C[Si](C)(C)OOC(=O)C(=O)C(=O)[N+](=O)[O-],"[[14, 56], [15, 144], [16, 18], [18, 0], [25, ...",neims_tms,[M]+,C6H9NO7Si,C[Si](C)(C)OOC(=O)C(=O)C(=O)[N+](=O)[O-],FPCOVUBGSLBMMO-UHFFFAOYSA-N,simulated,gecko_tms_6,<rdkit.Chem.rdchem.Mol object at 0x7fba2c82f890>,235.014828
4,C[Si](C)(C)OC(=O)C(O[Si](C)(C)C)C(=O)C(=O)C(=O...,"[[14, 31], [15, 47], [18, 18], [25, 5], [26, 4...",neims_tms,[M]+,C12H19NO9Si2,C[Si](C)(C)OC(=O)C(O[Si](C)(C)C)C(=O)C(=O)C(=O...,ZARPPVOVFLKXIU-UHFFFAOYSA-N,simulated,gecko_tms_7,<rdkit.Chem.rdchem.Mol object at 0x7fba2c82f900>,377.059834
...,...,...,...,...,...,...,...,...,...,...,...
159598,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,"[[14, 5], [15, 17], [16, 0], [26, 3], [27, 49]...",neims_tms,[M]+,C16H32N2O15Si3,CC1(O[Si](C)(C)C)C(C(O[Si](C)(C)C)C(=O)OO[N+](...,JHFUNAOALBCUEU-UHFFFAOYSA-N,simulated,gecko_tms_166429,<rdkit.Chem.rdchem.Mol object at 0x7fba2b6ab890>,576.111048
159599,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,"[[2, 3], [8, 0], [14, 15], [15, 30], [18, 21],...",neims_tms,[M]+,C16H33NO11Si3,CC1(O[Si](C)(C)C)C(C(C=O)O[Si](C)(C)C)OOC1(O[S...,WIPWTSILCDGNDN-UHFFFAOYSA-N,simulated,gecko_tms_166430,<rdkit.Chem.rdchem.Mol object at 0x7fba2b6ab900>,499.136140
159600,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,"[[14, 18], [15, 44], [16, 1], [18, 1], [20, 5]...",neims_tms,[M]+,C13H23NO11Si2,CC1(O[Si](C)(C)C)C(C(=O)C=O)OOC1(O[Si](C)(C)C)...,TZYJPJSYLIRUID-UHFFFAOYSA-N,simulated,gecko_tms_166431,<rdkit.Chem.rdchem.Mol object at 0x7fba2b6ab970>,425.080964
159601,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,"[[14, 0], [15, 27], [26, 21], [27, 30], [28, 6...",neims_tms,[M]+,C12H22N2O14Si2,CC1(O[Si](C)(C)C)C(C(=O)OO[N+](=O)[O-])OOC1(O[...,FAZHSBRCQRQGLZ-UHFFFAOYSA-N,simulated,gecko_tms_166432,<rdkit.Chem.rdchem.Mol object at 0x7fba2b6ab9e0>,474.060956


#### Write spec to MGF file

In [ ]:
output_path = Path(f'{output_dir}/neims_gecko_TMS_spectra_{num_mols}.mgf')
write_mgf(df, output_path)

#### Write labels to .tsv

In [ ]:
neims_labels = df[['dataset', 'compound', 'ionization', 'formula', 'SMILES', 'inchikey', 'instrument']]
neims_labels.index = df['compound']
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_gecko_labels_{num_mols}.tsv', sep='\t')
    df['SMILES'].to_csv(f'{output_dir}/neims_gecko_smiles_{num_mols}.csv')
    df['SMILES'].to_csv(f'{output_dir}/neims_gecko_lookup_smiles_{num_mols}.txt', header=None, index=False)

In [ ]:
raise Exception

Exception: 

In [ ]:
path_to_data = f'../../../data/neims/Franklin_dataset'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)
fname = f'df_neims_franklin_3_9_22.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)
#df = df.sample(num_mols, random_state=seed)
df

,SMILES,spec
0,C=CC(=C)CCC=C(C)CCC=C(C)CCC=C(C)C,"[[26, 27], [27, 154], [28, 56], [29, 213], [30..."
1,CC(=O)C=CC1(C)C(=O)C2(CCC1(C)C)CO2,"[[25, 5], [26, 55], [27, 225], [28, 113], [29,..."
2,CC1(C)CCCC2(C)OC(=O)C=C12,"[[14, 2], [15, 13], [26, 102], [27, 418], [28,..."
3,Nc1ccc2[nH]c3ccccc3c2c1,"[[24, 2], [26, 5], [27, 39], [28, 29], [37, 27..."
4,CC(C)CCCC(C)CCCC1(C)CCC(=O)O1,"[[29, 42], [39, 49], [41, 447], [42, 182], [43..."
5,CC1(C)CCC2(C)CCC3(C)C(=CCC4C5(C)CCC(=O)C(C)(C)...,"[[40, 12], [55, 17], [56, 1], [67, 26], [69, 9..."
6,CC(C)=CCCC(C)=CCCC(C)=CCCC=C(C)CCC(C)=CCCC=O,"[[1, 0], [6, 0], [18, 9], [25, 3], [26, 37], [..."
7,C=C(C)C1CCC2(C)CCC3(C)C(CCC4C5(C)CCC(OC(C)=O)C...,"[[18, 6], [26, 21], [27, 141], [28, 99], [29, ..."
8,CCCCCCCCCCCCC1CCC(=O)O1,"[[26, 22], [27, 158], [28, 105], [29, 246], [3..."
9,CC1(C)C2CCC3(C2)C1C=CC(=O)C3(C)C,"[[14, 0], [15, 7], [26, 66], [27, 375], [28, 1..."


In [ ]:
labels_df = pd.read_csv(f'{path_to_data}/labels.tsv', sep='\t')
labels_df['compound'] = labels_df['spec']
labels_df = labels_df.drop(columns=['smiles', 'Unnamed: 0', 'spec'])
labels_df

,dataset,ionization,formula,inchikey,instrument,compound
0,franklin,[M]+,C20H32,XSIVJVJUIXOEPW-UHFFFAOYSA-N,simulated,franklin_0
1,franklin,[M]+,C14H20O3,QZPGDWIOTPDRKE-UHFFFAOYSA-N,simulated,franklin_1
2,franklin,[M]+,C11H16O2,IMKHDCBNRDRUEB-UHFFFAOYSA-N,simulated,franklin_2
3,franklin,[M]+,C12H10N2,LRSYZHFYNDZXMU-UHFFFAOYSA-N,simulated,franklin_3
4,franklin,[M]+,C16H30O2,JQTJGWFLEGISPH-UHFFFAOYSA-N,simulated,franklin_4
5,franklin,[M]+,C30H48O,LIIFBMGUDSHTOU-UHFFFAOYSA-N,simulated,franklin_5
6,franklin,[M]+,C27H44O,BTXPQHRTBWDBAD-UHFFFAOYSA-N,simulated,franklin_6
7,franklin,[M]+,C32H52O2,ODSSDTBFHAYYMD-UHFFFAOYSA-N,simulated,franklin_7
8,franklin,[M]+,C16H30O2,SRIFJCOBFTWCTM-UHFFFAOYSA-N,simulated,franklin_8
9,franklin,[M]+,C15H22O,RKGKLJHVQKIFJZ-UHFFFAOYSA-N,simulated,franklin_9


In [ ]:
df = pd.merge(df, labels_df, left_index=True, right_index=True)
df

,SMILES,spec,dataset,ionization,formula,inchikey,instrument,compound
0,C=CC(=C)CCC=C(C)CCC=C(C)CCC=C(C)C,"[[26, 27], [27, 154], [28, 56], [29, 213], [30...",franklin,[M]+,C20H32,XSIVJVJUIXOEPW-UHFFFAOYSA-N,simulated,franklin_0
1,CC(=O)C=CC1(C)C(=O)C2(CCC1(C)C)CO2,"[[25, 5], [26, 55], [27, 225], [28, 113], [29,...",franklin,[M]+,C14H20O3,QZPGDWIOTPDRKE-UHFFFAOYSA-N,simulated,franklin_1
2,CC1(C)CCCC2(C)OC(=O)C=C12,"[[14, 2], [15, 13], [26, 102], [27, 418], [28,...",franklin,[M]+,C11H16O2,IMKHDCBNRDRUEB-UHFFFAOYSA-N,simulated,franklin_2
3,Nc1ccc2[nH]c3ccccc3c2c1,"[[24, 2], [26, 5], [27, 39], [28, 29], [37, 27...",franklin,[M]+,C12H10N2,LRSYZHFYNDZXMU-UHFFFAOYSA-N,simulated,franklin_3
4,CC(C)CCCC(C)CCCC1(C)CCC(=O)O1,"[[29, 42], [39, 49], [41, 447], [42, 182], [43...",franklin,[M]+,C16H30O2,JQTJGWFLEGISPH-UHFFFAOYSA-N,simulated,franklin_4
5,CC1(C)CCC2(C)CCC3(C)C(=CCC4C5(C)CCC(=O)C(C)(C)...,"[[40, 12], [55, 17], [56, 1], [67, 26], [69, 9...",franklin,[M]+,C30H48O,LIIFBMGUDSHTOU-UHFFFAOYSA-N,simulated,franklin_5
6,CC(C)=CCCC(C)=CCCC(C)=CCCC=C(C)CCC(C)=CCCC=O,"[[1, 0], [6, 0], [18, 9], [25, 3], [26, 37], [...",franklin,[M]+,C27H44O,BTXPQHRTBWDBAD-UHFFFAOYSA-N,simulated,franklin_6
7,C=C(C)C1CCC2(C)CCC3(C)C(CCC4C5(C)CCC(OC(C)=O)C...,"[[18, 6], [26, 21], [27, 141], [28, 99], [29, ...",franklin,[M]+,C32H52O2,ODSSDTBFHAYYMD-UHFFFAOYSA-N,simulated,franklin_7
8,CCCCCCCCCCCCC1CCC(=O)O1,"[[26, 22], [27, 158], [28, 105], [29, 246], [3...",franklin,[M]+,C16H30O2,SRIFJCOBFTWCTM-UHFFFAOYSA-N,simulated,franklin_8
9,CC1(C)C2CCC3(C2)C1C=CC(=O)C3(C)C,"[[14, 0], [15, 7], [26, 66], [27, 375], [28, 1...",franklin,[M]+,C15H22O,RKGKLJHVQKIFJZ-UHFFFAOYSA-N,simulated,franklin_9


In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['inchikey'] = df['mols'].apply(MolToInchiKey)


In [ ]:
output_path = Path(f'{output_dir}/spectra.mgf')
write_mgf(df, output_path)

In [ ]:
neims_labels = df[['dataset', 'compound', 'ionization', 'formula', 'SMILES', 'inchikey', 'instrument']]
neims_labels.index = df['compound']
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/labels.tsv', sep='\t')
    df['SMILES'].to_csv(f'{output_dir}/smiles.csv')
    df['SMILES'].to_csv(f'{output_dir}/smiles.txt', header=None, index=False)

## Load canopus aug data

In [ ]:
path_to_data = f'../../../data/paired_spectra/canopus_train/aug_iceberg_canopus_train/canopus_hplus_100_0'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
canopus_data = utils.parse_spectra_mgf(f'{path_to_data}/full_out.mgf')

400000it [00:25, 15429.23it/s]


In [ ]:
canopus_subset = canopus_data[0:num_mols]
subset_str = utils.build_mgf_str(canopus_subset)
#np.savetxt('../data/paired_spectra/canopus_train/aug_iceberg_canopus_train/canopus_hplus_100_0/subset_out.mgf', subset_str)
if ENABLE_FILE_EXPORTS:
    with open(f'{output_dir}/aug_canopus_spectra_{num_mols}.mgf', 'w+') as file:
        file.writelines(subset_str)
subset_ids = []
for idx, el in enumerate(canopus_subset):
    subset_ids.append(el[0]['ID'])
subset_ids[0:5]

100%|██████████| 10000/10000 [00:01<00:00, 5310.60it/s]


['aug_89166', 'aug_72305', 'aug_70314', 'aug_137643', 'aug_19834']

In [ ]:
canopus_labels = pd.read_csv(f'../../../data/paired_spectra/canopus_train/aug_iceberg_canopus_train/biomols_filtered_smiles_canopus_train_labels.tsv', sep='\t')
canopus_labels = canopus_labels[canopus_labels['spec'].isin(subset_ids)]
ENABLE_FILE_EXPORTS= True
if ENABLE_FILE_EXPORTS:
    canopus_labels.to_csv(f'{output_dir}/aug_canopus_labels_{num_mols}.tsv', sep='\t')

In [ ]:
df_out = pd.DataFrame()
df_out['SMILES'] = canopus_labels['smiles'].values
if ENABLE_FILE_EXPORTS:
    df_out.to_csv(f'{output_dir}/aug_canopus_smiles_{num_mols}.csv')

## Exp canopus

In [ ]:
path_to_data = f'../../../data/paired_spectra/canopus_train/'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/exp_canopus_train/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
files = ['../../../data/paired_spectra/canopus_train/splits/canopus_hplus_100_0.tsv',
         '../../../data/paired_spectra/canopus_train/splits/canopus_hplus_100_1.tsv',
         '../../../data/paired_spectra/canopus_train/splits/canopus_hplus_100_2.tsv']
dfs = []
for file in files:
    split = pd.read_csv(file, sep='\t')
    test_split = split[split['split'] == 'test']
    dfs.append(test_split)
splits = pd.concat(dfs)
splits_all_test = splits.drop_duplicates(subset='name').drop(columns=['split'])
if ENABLE_FILE_EXPORTS:
    splits_all_test.to_csv(f'{output_dir}/exp_canopus_all_test_labels.tsv', sep='\t', index=False)

In [ ]:
tests = splits_all_test['name'].values
tests
specs = []
for measurement in tests:
    file = f'{path_to_data}/spec_files/{measurement}.ms'
    spec = utils.parse_spectra(file)
    specs.append(spec)
tests.shape

(2206,)

In [ ]:
mgf_str = utils.build_mgf_str(specs)

100%|██████████| 2206/2206 [00:04<00:00, 520.63it/s]


In [ ]:
ENABLE_FILE_EXPORTS = True
if ENABLE_FILE_EXPORTS:
    with open(f'{output_dir}/exp_canopus_all_test_spectra.mgf', 'w+') as file:
        file.writelines(mgf_str)

In [ ]:
labels = pd.read_csv(f'{path_to_data}/labels.tsv', sep='\t')
labels = labels[labels['spec'].isin(tests)]
if ENABLE_FILE_EXPORTS:
    labels.to_csv(f'{output_dir}/exp_canopus_all_test_labels.tsv', sep='\t')

In [ ]:
df_out = pd.DataFrame()
df_out['SMILES'] = labels['smiles'].values
ENABLE_FILE_EXPORTS = True
if ENABLE_FILE_EXPORTS:
    df_out.to_csv(f'{output_dir}/exp_canopus_all_test_smiles.csv')

### Franklin neims

In [ ]:
path_to_data = f'../../../data/neims/Franklin_dataset'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
fname = f'{path_to_data}/df_neims_franklin_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'neims_fr'
df['compound'] = 'neims_fr' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'simulated'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CC(C)=CCC/C(C)=C/CC/C(C)=C/CCC(=C)C=C,"[[26, 27], [27, 154], [28, 56], [29, 213], [30...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab9a0>,C20H32,[M]+,neims_fr,neims_fr0,272.250401,simulated
1,CC(=O)C=CC1(C)C(=O)C2(CCC1(C)C)CO2,"[[25, 5], [26, 55], [27, 225], [28, 113], [29,...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab820>,C14H20O3,[M]+,neims_fr,neims_fr1,236.141244,simulated
2,CC1(C)CCC[C@@]2(C)OC(=O)C=C12,"[[14, 2], [15, 13], [26, 102], [27, 418], [28,...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab760>,C11H16O2,[M]+,neims_fr,neims_fr2,180.115030,simulated
3,NC1=CC2=C(C=C1)NC3=C2C=CC=C3,"[[24, 2], [26, 5], [27, 39], [28, 29], [37, 27...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab460>,C12H10N2,[M]+,neims_fr,neims_fr3,182.084398,simulated
4,CC(C)CCCC(C)CCCC1(C)CCC(=O)O1,"[[29, 42], [39, 49], [41, 447], [42, 182], [43...",<rdkit.Chem.rdchem.Mol object at 0x7efde38ab5e0>,C16H30O2,[M]+,neims_fr,neims_fr4,254.224580,simulated
...,...,...,...,...,...,...,...,...,...
64,C[Si](C)(C)OC(=O)C1CCC(=O)N1,"[[14, 8], [15, 68], [16, 3], [18, 32], [26, 63...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610e20>,C8H15NO3Si,[M]+,neims_fr,neims_fr64,201.082120,simulated
65,CCCOC(=O)C1=CC=CC=C1C(=O)OCC(C)CC,"[[27, 96], [28, 10], [29, 111], [39, 69], [41,...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610b80>,C16H22O4,[M]+,neims_fr,neims_fr65,278.151809,simulated
66,C[Si](C)(C)OC(=O)C1=CC=CC=C1C(=O)O[Si](C)(C)C,"[[14, 1], [15, 86], [18, 0], [29, 112], [30, 0...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610ca0>,C14H22O4Si2,[M]+,neims_fr,neims_fr66,310.105662,simulated
67,COC1=CC(C(CO[Si](C)(C)C)O[Si](C)(C)C)=CC=C1O[S...,"[[29, 18], [41, 10], [43, 59], [44, 71], [45, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde3610b20>,C18H36O4Si3,[M]+,neims_fr,neims_fr67,400.192139,simulated


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_franklin_labels.tsv', sep='\t')

In [ ]:
output_path = Path(f'{output_dir}/neims_franklin_spectra.mgf')
write_mgf(df, output_path)

### Franklin exp

In [ ]:
fname = f'{path_to_data}/df_exp_franklin_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'exp_fr'
df['compound'] = 'exp_fr' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'unknown'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CC(C)=CCC/C(C)=C/CC/C(C)=C/CCC(=C)C=C,"[[41, 543], [42, 29], [43, 80], [53, 76], [54,...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c537c0>,C20H32,[M]+,exp_fr,exp_fr0,272.250401,unknown
1,CC(=O)C=CC1(C)C(=O)C2(CCC1(C)C)CO2,"[[26, 7], [27, 94], [28, 23], [29, 67], [30, 3...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c53ac0>,C14H20O3,[M]+,exp_fr,exp_fr1,236.141244,unknown
2,CC1(C)CCC[C@@]2(C)OC(=O)C=C12,"[[38, 3], [39, 59], [40, 19], [41, 106], [42, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c53700>,C11H16O2,[M]+,exp_fr,exp_fr2,180.115030,unknown
3,NC1=CC2=C(C=C1)NC3=C2C=CC=C3,"[[27, 5], [28, 14], [38, 3], [39, 10], [41, 4]...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c535e0>,C12H10N2,[M]+,exp_fr,exp_fr3,182.084398,unknown
4,CC(C)CCCC(C)CCCC1(C)CCC(=O)O1,"[[39, 31], [40, 6], [41, 175], [42, 43], [43, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde8c53580>,C16H30O2,[M]+,exp_fr,exp_fr4,254.224580,unknown
...,...,...,...,...,...,...,...,...,...
64,C[Si](C)(C)OC(=O)C1CCC(=O)N1,"[[15, 2], [26, 3], [27, 13], [28, 72], [29, 15...",<rdkit.Chem.rdchem.Mol object at 0x7efde3606d00>,C8H15NO3Si,[M]+,exp_fr,exp_fr64,201.082120,unknown
65,CCCOC(=O)C1=CC=CC=C1C(=O)OCC(C)CC,"[[27, 10], [29, 11], [39, 7], [41, 28], [42, 1...",<rdkit.Chem.rdchem.Mol object at 0x7efde3606b80>,C16H22O4,[M]+,exp_fr,exp_fr65,278.151809,unknown
66,C[Si](C)(C)OC(=O)C1=CC=CC=C1C(=O)O[Si](C)(C)C,"[[25, 2], [26, 6], [27, 4], [29, 2], [31, 1], ...",<rdkit.Chem.rdchem.Mol object at 0x7efde36069a0>,C14H22O4Si2,[M]+,exp_fr,exp_fr66,310.105662,unknown
67,COC1=CC(C(CO[Si](C)(C)C)O[Si](C)(C)C)=CC=C1O[S...,"[[40, 17], [43, 13], [44, 19], [45, 94], [46, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde3606b20>,C18H36O4Si3,[M]+,exp_fr,exp_fr67,400.192139,unknown


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/exp_franklin_labels.tsv', sep='\t')

In [ ]:
output_path = Path(f'{output_dir}/exp_franklin_spectra.mgf')
write_mgf(df, output_path)

## canopus neims

In [ ]:
path_to_data = f'../../../data/neims/mist'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)

In [ ]:
fname = f'{path_to_data}/df_neims_canopus_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'canopus_neims'
df['compound'] = 'canopus_neims_' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'simulated'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CCC(=O)CCCCCC1NC(=O)C2CCCCN2C(=O)C(Cc2ccccc2)N...,"[[28, 9], [30, 3], [32, 55], [36, 97], [40, 28...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e69a0>,C28H40N4O5,[M]+,canopus_neims,canopus_neims_0,512.299870,simulated
1,CCCCCCCCCCCCCC(=O)NC(C)(C)C(=O)N1CCCC1C(=O)NC(...,"[[28, 7], [30, 54], [39, 16], [43, 69], [44, 3...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6ac0>,C45H83N7O8,[M]+,canopus_neims,canopus_neims_1,849.630313,simulated
2,CC(C)CC1NC(=O)C(NC(=O)c2ncccc2O)C(C)OC(=O)C(c2...,"[[30, 43], [39, 338], [40, 159], [41, 761], [4...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6a00>,C44H62N8O11,[M]+,canopus_neims,canopus_neims_2,878.453805,simulated
3,COc1cc(C2C3(O)C(O)C4CC2(O)C(O)(C(=O)O4)C3C(=O)...,"[[25, 1], [26, 30], [27, 65], [28, 104], [29, ...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6b20>,C22H20O10,[M]+,canopus_neims,canopus_neims_3,444.105647,simulated
4,COC1C=COC2(C)Oc3c(C)c(O)c4c(c3C2=O)C(=O)C=C(NC...,"[[25, 9], [30, 21], [32, 15], [33, 9], [38, 7]...",<rdkit.Chem.rdchem.Mol object at 0x7efde85e6a60>,C37H45NO12,[M]+,canopus_neims,canopus_neims_4,695.294176,simulated
...,...,...,...,...,...,...,...,...,...
2201,CCOc1ccc2oc(=O)c(-c3ccc(O)cc3)c(C)c2c1,"[[0, 0], [14, 12], [15, 47], [16, 4], [17, 2],...",<rdkit.Chem.rdchem.Mol object at 0x7efde33abac0>,C18H16O4,[M]+,canopus_neims,canopus_neims_2201,296.104859,simulated
2202,Cc1cn(C2OC(CO)C(O)C2O)c(=O)nc1N,"[[14, 38], [15, 122], [16, 32], [17, 52], [18,...",<rdkit.Chem.rdchem.Mol object at 0x7efde33aba60>,C10H15N3O5,[M]+,canopus_neims,canopus_neims_2202,257.101171,simulated
2203,COc1cc(OC)c(C(=O)C=Cc2ccc(OC)c(OC)c2)c(OC)c1,"[[14, 2], [15, 162], [26, 46], [27, 123], [28,...",<rdkit.Chem.rdchem.Mol object at 0x7efde33abb20>,C20H22O6,[M]+,canopus_neims,canopus_neims_2203,358.141638,simulated
2204,O=C1NCCc2c1[nH]c1c(Cl)cccc21,"[[33, 2], [35, 16], [36, 43], [37, 37], [38, 7...",<rdkit.Chem.rdchem.Mol object at 0x7efde33abbe0>,C11H9ClN2O,[M]+,canopus_neims,canopus_neims_2204,220.040341,simulated


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_canopus_labels.tsv', sep='\t')

output_path = Path(f'{output_dir}/neims_canopus_spectra.mgf')
write_mgf(df, output_path)

## canopus aug neims

In [ ]:
fname = f'{path_to_data}/df_neims_aug_3_8_18.pkl'
with open(fname, 'rb') as file:
    df = pkl.load(file)

In [ ]:
df['mols'] = df['SMILES'].apply(MolFromSmiles)
df['formula'] = df['mols'].apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'aug_neims'
df['compound'] = 'aug_neims_' + df.index.astype(str)
df['parentmass'] = df['mols'].apply(ExactMolWt)
df['parentmass'] = df['parentmass']
df['instrument'] = 'simulated'
df

,SMILES,spec,mols,formula,ionization,dataset,compound,parentmass,instrument
0,CCCCCCCCCCCCCCCCOCC(COP(=O)([O-])OCC[N+](C)(C)...,"[[25, 24], [26, 109], [27, 238], [28, 168], [2...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff73880>,C46H82NO7P,[M]+,aug_neims,aug_neims_0,791.582891,simulated
1,CC1=C(C(=O)OC2=C1C=CC(=C2)OC(C)C(=O)NCCC(=O)O)C,"[[29, 67], [33, 58], [34, 27], [36, 32], [38, ...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff738e0>,C17H19NO6,[M]+,aug_neims,aug_neims_1,333.121237,simulated
2,CC(C)C1=CC=C(C=C1)CN2CCC(C2)N(C)S(=O)(=O)C3=CC...,"[[15, 6], [27, 5], [28, 98], [30, 260], [33, 5...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff73940>,C22H30N2O4S2,[M]+,aug_neims,aug_neims_2,450.164699,simulated
3,CCN1C2=C(C=C(C=C2)S(=O)(=O)N(C)C)N=C1CCC(=O)NC...,"[[33, 55], [34, 75], [36, 188], [37, 77], [38,...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff739a0>,C20H23N5O5S,[M]+,aug_neims,aug_neims_3,445.141990,simulated
4,CCC(C(=O)NC1=C(C=CC(=C1)Cl)Cl)N(C2=CC=CC=C2)S(...,"[[33, 32], [34, 1], [35, 98], [36, 149], [37, ...",<rdkit.Chem.rdchem.Mol object at 0x7efdbff73a00>,C17H18Cl2N2O3S,[M]+,aug_neims,aug_neims_4,400.041519,simulated
...,...,...,...,...,...,...,...,...,...
9995,CN(C)CCN1C=C(C=N1)NC(=O)CCCC2=NC(=NO2)C3=CC=C(...,"[[33, 8], [36, 45], [37, 20], [38, 39], [39, 9...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb956a0>,C19H23ClN6O2,[M]+,aug_neims,aug_neims_9995,402.157102,simulated
9996,CC(C(=O)NC(CC1=CN=CN1)C(=O)O)NC(=O)C(CCCN=C(N)...,"[[16, 72], [17, 72], [18, 74], [25, 0], [28, 5...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb95700>,C24H35N9O6,[M]+,aug_neims,aug_neims_9996,545.271030,simulated
9997,CCC(C(=O)OC)SC1=NN=C(S1)NC(=O)C2=NOC(=C2)C3=CC...,"[[29, 64], [33, 105], [34, 51], [35, 20], [36,...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb95760>,C15H14N4O5S2,[M]+,aug_neims,aug_neims_9997,394.040562,simulated
9998,CC1(C2CC=C(C1C2)C(=O)O)C,"[[14, 5], [16, 13], [17, 5], [26, 102], [27, 4...",<rdkit.Chem.rdchem.Mol object at 0x7efdbfb957c0>,C10H14O2,[M]+,aug_neims,aug_neims_9998,166.099380,simulated


In [ ]:
neims_labels = df[['formula', 'ionization', 'dataset', 'compound', 'parentmass', 'instrument']]
neims_labels.index.name = 'spec'    
if ENABLE_FILE_EXPORTS:
    neims_labels.to_csv(f'{output_dir}/neims_aug_labels.tsv', sep='\t')

output_path = Path(f'{output_dir}/neims_aug_spectra.mgf')
write_mgf(df, output_path)

# LINDSAY YEE DATASET

In [18]:
path_to_data = f'../../../data/yee/'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)
fname = f"{path_to_data}/df_lindsay_test.pkl"
with open(fname, 'rb') as file:
    df: pd.DataFrame = pkl.load(file)
df.head(3)

,Index,Name,Synon,Campaign/Experimental_Source,Experimental_Conditions,UID,Contributor_ID,Comments,Suspected_Matches,Confirmed_Matches,...,Oven_temp,Publications,Contributor,Date_of_Entry,Num Peaks,5_Highest_Intensity_MZ,Base_Peak,MS,X-Values,Y-Values
1604,1619,"erythro 2-methyltetrol, 4TMS",NaN,Laboratory Oxidation,ER649GF2 (a-cedrene + NOx + hv),LDY-1619,NaN,Very likely contamination from a prior isopren...,NaN,NaN,...,40 C to 320 C at 3.5 C/min,"Yee, L. D., Isaacman-Vanwertz, G., Wernis, R. ...",Lindsay Yee,11/16/2020,160.0,"[219.0, 73.0, 117.0, 129.0, 147.0]",219.0,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...
1968,1988,"threo 2-methyltetrol, 4 TMS",NaN,Laboratory Oxidation,ER649GF2 (a-cedrene + NOx + hv),LDY-1988,NaN,Very likely contamination from a prior isopren...,NaN,NaN,...,40 C to 320 C at 3.5 C/min,"Yee, L. D., Isaacman-Vanwertz, G., Wernis, R. ...",Lindsay Yee,11/16/2020,127.0,"[73.0, 219.0, 117.0, 129.0, 147.0]",73.0,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...
3709,3831,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...","beta-nocaryophyllonic acid, TMS",Laboratory Oxidation,ER720GF3 (a-humulene + O3),LDY-3831,NaN,Contaminant from a previous beta-caryophyllene...,NaN,NaN,...,40 C to 320 C at 3.5 C/min,"Yee, L. D., Isaacman-Vanwertz, G., Wernis, R. ...",Lindsay Yee,11/16/2020,248.0,"[75.0, 73.0, 43.0, 185.0, 137.0]",75.0,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...


In [19]:
print(df.columns)

Index(['Index', 'Name', 'Synon', 'Campaign/Experimental_Source',
       'Experimental_Conditions', 'UID', 'Contributor_ID', 'Comments',
       'Suspected_Matches', 'Confirmed_Matches', 'Predicted_Matches',
       'Compound_Structural_Info', 'Formula', 'MW', 'MW_Prediction',
       'ExactMass', 'VUV_Exact_Mass', 'CAS#', 'Ch3MS-RF_Cnum.p',
       'Ch3MS-RF_OC.p', 'Ch3MS-RF_OSc.p', 'Ch3MS-RF_VP.p', 'Column_Type',
       'Retention_index', 'd_alkane_RTI', 'n_alkane_RTI', 'Instrument',
       'Ionization', 'Injection_method', 'Derivatization_Agent', 'GC_column',
       'Oven_temp', 'Publications', 'Contributor', 'Date_of_Entry',
       'Num Peaks', '5_Highest_Intensity_MZ', 'Base_Peak', 'MS', 'X-Values',
       'Y-Values'],
      dtype='object')


In [20]:
df: pd.DataFrame = df.dropna(axis=1, how='all').dropna(axis=0, how='any', subset=['X-Values', 'Y-Values'])
df  = df[['Name', 'UID', 'MS', 'X-Values', 'Y-Values', 'Formula', 'Instrument']]
df = df[~df['Formula'].str.contains('D')]
df['compound'] = df['UID']
df.head(3)

,Name,UID,MS,X-Values,Y-Values,Formula,Instrument,compound
1604,"erythro 2-methyltetrol, 4TMS",LDY-1619,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1619
1968,"threo 2-methyltetrol, 4 TMS",LDY-1988,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1988
3709,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-3831,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...,C17H30O4Si,GCxGC HRToFMS,LDY-3831


In [21]:
yee_mz = df['X-Values'].values
yee_int = df['Y-Values'].values
def raw_str_to_mz_int(mz: str, inten: str, out_dtype = np.uint16):
    mz_preparsed = mz.replace('[', '').replace(']', '').replace('\n', '').split('.')
    inten_preparsed = inten.replace('[', '').replace(']', '').replace('\n', '').split('.')
    mz_preparsed = [str_in.strip() for str_in in mz_preparsed if str_in != '']
    inten_preparsed = [str_in.strip() for str_in in inten_preparsed if str_in != '']
    assert all([str_in.isnumeric() for str_in in mz_preparsed]), 'mz must be numeric string'
    assert all([str_in.isnumeric() for str_in in inten_preparsed]), 'intensity must be numeric string'
    X_arr = np.array(mz_preparsed, dtype=out_dtype)
    Y_arr = np.array(inten_preparsed, dtype=out_dtype)
    assert X_arr.shape == Y_arr.shape, \
        'mz and intensity dimensions must match'
    assert X_arr.shape[0] > 0 and Y_arr.shape[0] > 0, \
        'mz and intensity data must contain at least one element'
    assert all(X_arr > 0) and all(Y_arr >= 0), \
        'mz must be positive and intensities must be non-negative'
    assert all(X_arr < 10_000), 'mz cannot be arbitrarily large'
    output = np.array([*zip(X_arr, Y_arr)])
    assert output.shape[1] == 2, 'mz-inten pair array must be of shape (X, 2)'
    return output

yee_mz_int = []
for idx, (mz, inten) in enumerate(zip(yee_mz, yee_int)):
    yee_mz_int.append(raw_str_to_mz_int(mz, inten))
yee_mz_int

[array([[ 29,   2],
        [ 30,   1],
        [ 31,   2],
        [ 39,   3],
        [ 41,   7],
        [ 42,   1],
        [ 43,  15],
        [ 44,   9],
        [ 45,  61],
        [ 46,   2],
        [ 47,   5],
        [ 48,   1],
        [ 53,   2],
        [ 55,   2],
        [ 56,   1],
        [ 57,   8],
        [ 58,   8],
        [ 59,  28],
        [ 60,   2],
        [ 61,   5],
        [ 64,   1],
        [ 65,   3],
        [ 66,   2],
        [ 67,   2],
        [ 68,   3],
        [ 69,  30],
        [ 70,   1],
        [ 71,   1],
        [ 72,   8],
        [ 73, 983],
        [ 74,  53],
        [ 75, 209],
        [ 76,  10],
        [ 77,   9],
        [ 78,   1],
        [ 79,   3],
        [ 80,   3],
        [ 81,   1],
        [ 82,   3],
        [ 83,   9],
        [ 85,   1],
        [ 87,   4],
        [ 89,  13],
        [ 90,   3],
        [ 91,   1],
        [ 92,   1],
        [ 93,   5],
        [ 96,   1],
        [ 97,   1],
        [ 98,   2],


In [36]:
df['spec'] = yee_mz_int
df.head(3)

,Name,UID,MS,X-Values,Y-Values,Formula,Instrument,compound,spec,parentmass,dataset,spectrum,formula,instrument,smiles,inchikey,ionization,SMILES
1604,"erythro 2-methyltetrol, 4TMS",LDY-1619,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1619,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ...",424.875,yee,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ...",C17H44O4Si4,GCxGC HRToFMS,na,na,[M]+,na
1968,"threo 2-methyltetrol, 4 TMS",LDY-1988,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1988,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2...",424.875,yee,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2...",C17H44O4Si4,GCxGC HRToFMS,na,na,[M]+,na
3709,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-3831,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...,C17H30O4Si,GCxGC HRToFMS,LDY-3831,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],...",326.508,yee,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],...",C17H30O4Si,GCxGC HRToFMS,na,na,[M]+,na


In [23]:
import re
from collections import defaultdict

def parse_formula(formula):
    # Remove leading junk like '?'
    formula = formula.lstrip('?')
    
    # Regex: element symbol (capital + optional lowercase), optional number
    tokens = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    
    composition = defaultdict(int)
    
    for element, count in tokens:
        composition[element] += int(count) if count else 1
    
    return dict(composition)
parsed = [parse_formula(f) for f in df['Formula'].unique()]
parsed
elements = set()
for el in parsed:
    for key, val in el.items():
        elements.add(key)
elements

{'C', 'H', 'N', 'O', 'P', 'S', 'Si'}

In [24]:
df_atoms = pd.DataFrame([parse_formula(f) for f in df['Formula'].values]).fillna(0).astype(int)
df_atoms

,C,H,O,Si,P,S,N
0,17,44,4,4,0,0,0
1,17,44,4,4,0,0,0
2,17,30,4,1,0,0,0
3,17,30,4,1,0,0,0
4,15,30,4,2,0,0,0
...,...,...,...,...,...,...,...
472,28,56,2,0,0,0,0
473,30,62,0,0,0,0,0
474,30,50,0,0,0,0,0
475,27,44,0,0,0,0,0


In [25]:
weights = np.array([12.011, 1.0080, 15.999, 28.085, 30.974, 32.06, 14.007])
mol_wt_formula = np.sum(df_atoms.values * weights, axis=1)
mol_wt_formula

array([424.875, 424.875, 326.508, 326.508, 330.571, 256.417, 344.598,
       318.516, 262.452, 168.236, 256.417, 314.54 , 336.65 , 222.24 ,
       384.841, 276.479, 308.64 , 248.425, 330.571, 519.078, 350.633,
       274.463, 230.423, 168.236, 216.396, 194.305, 392.714, 330.571,
       256.417, 344.598, 262.452, 222.24 , 168.236, 276.479, 350.633,
       330.571, 256.417, 392.714, 236.355, 398.69 , 324.536, 238.327,
       326.508, 326.508, 324.536, 236.355, 326.508, 324.536, 298.542,
       264.352, 322.623, 240.392, 287.451, 242.393, 278.348, 350.675,
       458.848, 560.981, 314.528, 164.204, 236.386, 370.667, 364.702,
       270.457, 260.524, 224.331, 190.245, 312.512, 310.54 , 296.438,
       276.479, 196.321, 234.298, 234.326, 273.479, 294.541, 192.261,
       268.503, 364.66 , 502.899, 268.503, 270.407, 324.567, 326.539,
       414.764, 304.536, 192.261, 286.459, 513.056, 496.98 , 178.234,
       396.703, 254.357, 555.093, 378.687, 352.634, 198.293, 378.687,
       502.754, 240.

In [26]:
df.head(3)

,Name,UID,MS,X-Values,Y-Values,Formula,Instrument,compound,spec
1604,"erythro 2-methyltetrol, 4TMS",LDY-1619,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1619,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ..."
1968,"threo 2-methyltetrol, 4 TMS",LDY-1988,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1988,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2..."
3709,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-3831,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...,C17H30O4Si,GCxGC HRToFMS,LDY-3831,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],..."


In [27]:
output_path = Path(f'{output_dir}/yee_test_{num_mols}.mgf')
df['parentmass'] = mol_wt_formula
df['dataset'] = 'yee'
write_mgf(df, output_path)

In [28]:
df['spectrum'] = df['spec'] 
df['formula'] = df['Formula']
df['instrument'] = df['Instrument']
df['smiles'] = 'na'
df['inchikey'] = 'na'
df['spec'] = df['UID']
df.head(3)

,Name,UID,MS,X-Values,Y-Values,Formula,Instrument,compound,spec,parentmass,dataset,spectrum,formula,instrument,smiles,inchikey
1604,"erythro 2-methyltetrol, 4TMS",LDY-1619,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1619,LDY-1619,424.875,yee,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ...",C17H44O4Si4,GCxGC HRToFMS,na,na
1968,"threo 2-methyltetrol, 4 TMS",LDY-1988,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1988,LDY-1988,424.875,yee,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2...",C17H44O4Si4,GCxGC HRToFMS,na,na
3709,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-3831,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...,C17H30O4Si,GCxGC HRToFMS,LDY-3831,LDY-3831,326.508,yee,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],...",C17H30O4Si,GCxGC HRToFMS,na,na


In [29]:
import csv

In [30]:
df['ionization'] = '[M]+'

In [31]:
df_labels = df[['dataset', 'spec', 'ionization', 'formula', 'smiles', 'inchikey', 'instrument']]
df_labels.reset_index().drop(columns='index').to_csv('yee_labels.tsv', sep='\t', quoting=csv.QUOTE_ALL)

In [32]:
df_labels.head(3)

,dataset,spec,ionization,formula,smiles,inchikey,instrument
1604,yee,LDY-1619,[M]+,C17H44O4Si4,na,na,GCxGC HRToFMS
1968,yee,LDY-1988,[M]+,C17H44O4Si4,na,na,GCxGC HRToFMS
3709,yee,LDY-3831,[M]+,C17H30O4Si,na,na,GCxGC HRToFMS


In [33]:
import pickle

In [34]:
df['SMILES'] = df['smiles']
df

,Name,UID,MS,X-Values,Y-Values,Formula,Instrument,compound,spec,parentmass,dataset,spectrum,formula,instrument,smiles,inchikey,ionization,SMILES
1604,"erythro 2-methyltetrol, 4TMS",LDY-1619,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1619,LDY-1619,424.875,yee,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ...",C17H44O4Si4,GCxGC HRToFMS,na,na,[M]+,na
1968,"threo 2-methyltetrol, 4 TMS",LDY-1988,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1988,LDY-1988,424.875,yee,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2...",C17H44O4Si4,GCxGC HRToFMS,na,na,[M]+,na
3709,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-3831,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...,C17H30O4Si,GCxGC HRToFMS,LDY-3831,LDY-3831,326.508,yee,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],...",C17H30O4Si,GCxGC HRToFMS,na,na,[M]+,na
5241,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-5380,29 14.0 ; 31 7.0 ; 33 3.0 ; 37 1.0 ; 38 5.0 ; ...,[ 29. 31. 33. 37. 38. 39. 40. 41. 42. ...,[ 14. 7. 3. 1. 5. 48. 12. 146. 23. ...,C17H30O4Si,GCxGC HRToFMS,LDY-5380,LDY-5380,326.508,yee,"[[29, 14], [31, 7], [33, 3], [37, 1], [38, 5],...",C17H30O4Si,GCxGC HRToFMS,na,na,[M]+,na
6753,"pinic acid, 2TMS",LDY-6914,29 2.0 ; 39 9.0 ; 40 1.0 ; 41 24.0 ; 42 3.0 ; ...,[ 29. 39. 40. 41. 42. 43. 44. 45. 46. ...,[ 2. 9. 1. 24. 3. 19. 9. 77. 5. ...,C15H30O4Si2,GCxGC HRToFMS,LDY-6914,LDY-6914,330.571,yee,"[[29, 2], [39, 9], [40, 1], [41, 24], [42, 3],...",C15H30O4Si2,GCxGC HRToFMS,na,na,[M]+,na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20678,myristic myristate,LDY-21890,30 16.0 ; 31 13.0 ; 32 999.0 ; 33 38.0 ; 37 13...,[ 30. 31. 32. 33. 37. 39. 42. 47. 48. ...,[ 16. 13. 999. 38. 13. 10. 10. 70. 86. ...,C28H56O2,GCxGC HRToFMS,LDY-21890,LDY-21890,424.754,yee,"[[30, 16], [31, 13], [32, 999], [33, 38], [37,...",C28H56O2,GCxGC HRToFMS,na,na,[M]+,na
20681,C30 n-alkane,LDY-21893,29 70.0 ; 31 34.0 ; 32 999.0 ; 34 88.0 ; 35 32...,[ 29. 31. 32. 34. 35. 39. 41. 43. 45. ...,[ 70. 34. 999. 88. 32. 68. 205. 300. 120. ...,C30H62,GCxGC HRToFMS,LDY-21893,LDY-21893,422.826,yee,"[[29, 70], [31, 34], [32, 999], [34, 88], [35,...",C30H62,GCxGC HRToFMS,na,na,[M]+,na
20682,squalene,LDY-21894,31 80.0 ; 32 999.0 ; 34 7.0 ; 35 22.0 ; 36 58....,[ 31. 32. 34. 35. 36. 37. 39. 40. 41. ...,[ 80. 999. 7. 22. 58. 18. 60. 308. 398. ...,C30H50,GCxGC HRToFMS,LDY-21894,LDY-21894,410.730,yee,"[[31, 80], [32, 999], [34, 7], [35, 22], [36, ...",C30H50,GCxGC HRToFMS,na,na,[M]+,na
20686,"cholesta-3,4-diene",LDY-21898,29 210.0 ; 30 27.0 ; 31 82.0 ; 32 999.0 ; 33 4...,[ 29. 30. 31. 32. 33. 37. 39. 41. 44. ...,[210. 27. 82. 999. 4. 22. 151. 92. 210. ...,C27H44,GCxGC HRToFMS,LDY-21898,LDY-21898,368.649,yee,"[[29, 210], [30, 27], [31, 82], [32, 999], [33...",C27H44,GCxGC HRToFMS,na,na,[M]+,na


In [37]:
df['spec'] = yee_mz_int
df.head(3)

,Name,UID,MS,X-Values,Y-Values,Formula,Instrument,compound,spec,parentmass,dataset,spectrum,formula,instrument,smiles,inchikey,ionization,SMILES
1604,"erythro 2-methyltetrol, 4TMS",LDY-1619,29 2.0 ; 30 1.0 ; 31 2.0 ; 39 3.0 ; 41 7.0 ; 4...,[ 29. 30. 31. 39. 41. 42. 43. 44. 45. ...,[ 2. 1. 2. 3. 7. 1. 15. 9. 61. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1619,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ...",424.875,yee,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ...",C17H44O4Si4,GCxGC HRToFMS,na,na,[M]+,na
1968,"threo 2-methyltetrol, 4 TMS",LDY-1988,29 7.0 ; 31 16.0 ; 39 12.0 ; 40 14.0 ; 41 2.0 ...,[ 29. 31. 39. 40. 41. 43. 44. 45. 46. ...,[ 7. 16. 12. 14. 2. 8. 4. 98. 12. ...,C17H44O4Si4,GCxGC HRToFMS,LDY-1988,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2...",424.875,yee,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2...",C17H44O4Si4,GCxGC HRToFMS,na,na,[M]+,na
3709,"trimethylsilyl 4-((1S,2R)-3,3-dimethyl-2-(3-ox...",LDY-3831,29 19.0 ; 30 5.0 ; 31 6.0 ; 34 2.0 ; 36 1.0 ; ...,[ 29. 30. 31. 34. 36. 38. 39. 40. 41. ...,[ 19. 5. 6. 2. 1. 2. 41. 14. 145. ...,C17H30O4Si,GCxGC HRToFMS,LDY-3831,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],...",326.508,yee,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],...",C17H30O4Si,GCxGC HRToFMS,na,na,[M]+,na


In [39]:
with open('yee_spec_inference.pkl', 'wb') as f:
    pickle.dump(df[['UID', 'parentmass', 'formula', 'ionization', 'instrument', 'spec']], f)
df[['UID', 'parentmass', 'formula', 'ionization', 'instrument', 'spec']].head(3)

,UID,parentmass,formula,ionization,instrument,spec
1604,LDY-1619,424.875,C17H44O4Si4,[M]+,GCxGC HRToFMS,"[[29, 2], [30, 1], [31, 2], [39, 3], [41, 7], ..."
1968,LDY-1988,424.875,C17H44O4Si4,[M]+,GCxGC HRToFMS,"[[29, 7], [31, 16], [39, 12], [40, 14], [41, 2..."
3709,LDY-3831,326.508,C17H30O4Si,[M]+,GCxGC HRToFMS,"[[29, 19], [30, 5], [31, 6], [34, 2], [36, 1],..."


In [40]:
df['spec'].iloc[0].shape

(160, 2)

# atmomaccs datasets

In [5]:
path_to_data = f'../../../data/neims/gecko_EIMS_spectra'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs'
os.makedirs(output_dir, exist_ok = True)
fname = f"{path_to_data}/df_neims_atmomaccs_3_9_22.pkl"
with open(fname, 'rb') as file:
    df = pkl.load(file)
df

,SMILES,spec,name
0,CCCCCCCCC=C,"[[15, 1], [25, 17], [26, 103], [27, 474], [28,...",ferraz-caetano_0
1,CCC(C)CC(C)(C)C,"[[15, 36], [25, 3], [26, 34], [27, 198], [28, ...",ferraz-caetano_1
2,CCCCCCCCC,"[[14, 9], [15, 34], [25, 3], [26, 85], [27, 44...",ferraz-caetano_2
3,CCCCCCCCCC,"[[14, 2], [15, 23], [25, 1], [26, 65], [27, 37...",ferraz-caetano_3
4,CC(C)CCC(C)(C)C,"[[14, 10], [15, 66], [25, 7], [26, 51], [27, 2...",ferraz-caetano_4
...,...,...,...
9384,CCCCC(=O)C(CC(=O)CC)O,"[[26, 18], [27, 131], [28, 66], [29, 253], [31...",wang_3409
9385,CC(=O)CC(C)(C)ON(=O)=O,"[[14, 33], [15, 88], [16, 10], [26, 47], [27, ...",wang_3410
9386,C(Cl)(Cl)(Cl)O,"[[14, 56], [16, 16], [17, 32], [18, 30], [19, ...",wang_3411
9387,CC1(C)CC(C1CCC(=O)OC)ON(=O)=O,"[[15, 14], [17, 4], [18, 11], [25, 18], [26, 9...",wang_3412


In [9]:
df[df['name'] == 'li_2000']

,SMILES,spec,name
4410,CC1=CC(=C(C(=C1)C)S(=O)(=O)OC2=CC=CC=C2)C,"[[17, 17], [18, 20], [27, 52], [28, 9], [32, 1...",li_2000


In [6]:
mols = df['SMILES'].apply(MolFromSmiles)
df['formula'] = mols.apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'atmomaccs'
df['compound'] = df['name']
df['parentmass'] = mols.apply(ExactMolWt)
df['inchikey'] = mols.apply(MolToInchiKey)
df['instrument'] = 'simulated'

In [7]:
rename_cols = {
    'SMILES': 'smiles',
    'name': 'spec'
}
df[['dataset', 'name', 'ionization', 'formula', 'SMILES', 'inchikey', 'instrument']].rename(columns=rename_cols).to_csv('labels_atmomaccs.tsv', sep='\t')

In [ ]:
output_dir = '../../../data/neims/atmomaccs/'
output_path = Path(f'{output_dir}/spectra.mgf')
write_mgf(df, output_path)

NameError: name 'write_mgf' is not defined

In [9]:
df['split'] = 'test'

In [10]:
split = df[['name', 'split']]

split.to_csv(Path(f'{output_dir}/split.tsv'), sep='\t')
split

,name,split
0,ferraz-caetano_0,test
1,ferraz-caetano_1,test
2,ferraz-caetano_2,test
3,ferraz-caetano_3,test
4,ferraz-caetano_4,test
...,...,...
9384,wang_3409,test
9385,wang_3410,test
9386,wang_3411,test
9387,wang_3412,test


# MSG NEIMS

In [5]:
path_to_data = f'../../../data/neims/gecko_EIMS_spectra'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs/msg'
os.makedirs(output_dir, exist_ok = True)
fname = f"{path_to_data}/df_neims_msg_3_9_22.pkl"
with open(fname, 'rb') as file:
    df = pkl.load(file)
df

,SMILES,spec,name
0,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1,"[[15, 57], [26, 12], [28, 37], [37, 31], [38, ...",msg-train_0
1,CC1CCCC(=O)CCCC=Cc2cc(O)cc(O)c2C(=O)O1,"[[17, 1], [26, 95], [27, 258], [28, 139], [29,...",msg-train_1
2,C=CC(C)(C)C1(CC2NC(=O)C(=Cc3cnc[nH]3)NC2=O)C(=...,"[[33, 39], [34, 28], [36, 90], [37, 75], [38, ...",msg-train_2
3,CC1CC2OC2C=CC(=O)CC(=O)O1,"[[14, 1], [15, 3], [25, 11], [26, 170], [27, 5...",msg-train_3
4,COC1=CC(=O)c2c(O)c3c(c(O)c2C1=O)CC1(C)OC(C(=O)...,"[[16, 4], [17, 33], [18, 70], [19, 3], [26, 17...",msg-train_4
...,...,...,...
28919,COP(=O)(OC)SCn1c(=O)oc2cc(Cl)cnc21,"[[33, 174], [34, 21], [35, 62], [36, 76], [37,...",msg-val_3179
28920,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2Cl)n1,"[[33, 38], [34, 29], [35, 50], [36, 88], [37, ...",msg-val_3180
28921,COC(=O)c1ccccc1S(=O)(=O)NC(=O)N(C)c1nc(C)nc(OC)n1,"[[28, 4], [38, 4], [39, 105], [40, 51], [41, 1...",msg-val_3181
28922,N#CC1(c2ccc(NC(=O)c3cccnc3NCc3ccncc3)cc2)CCCC1,"[[33, 51], [34, 17], [36, 39], [37, 45], [38, ...",msg-val_3182


In [6]:
mols = df['SMILES'].apply(MolFromSmiles)
df['formula'] = mols.apply(CalcMolFormula)
df['ionization'] = '[M]+'
df['dataset'] = 'msg_neims'
df['compound'] = df['name']
df['parentmass'] = mols.apply(ExactMolWt)
df['inchikey'] = mols.apply(MolToInchiKey)
df['instrument'] = 'simulated'

In [7]:
rename_cols = {
    'SMILES': 'smiles',
    'name': 'spec'
}
df[['dataset', 'name', 'ionization', 'formula', 'SMILES', 'inchikey', 'instrument']].rename(columns=rename_cols).to_csv(Path(f'{output_dir}/labels_msg.tsv'), sep='\t')

In [8]:
output_path = Path(f'{output_dir}/spectra.mgf')
write_mgf(df, output_path)

In [9]:
df.head(2)

,SMILES,spec,name,formula,ionization,dataset,compound,parentmass,inchikey,instrument
0,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1,"[[15, 57], [26, 12], [28, 37], [37, 31], [38, ...",msg-train_0,C16H17NO4,[M]+,msg_neims,msg-train_0,287.115758,VFMQMACUYWGDOJ-UHFFFAOYSA-N,simulated
1,CC1CCCC(=O)CCCC=Cc2cc(O)cc(O)c2C(=O)O1,"[[17, 1], [26, 95], [27, 258], [28, 139], [29,...",msg-train_1,C18H22O5,[M]+,msg_neims,msg-train_1,318.146724,MBMQEIFVQACCCH-UHFFFAOYSA-N,simulated


In [10]:
df['split'] = np.select(
    [
        df['compound'].str.contains('test', na=False),
        df['compound'].str.contains('val', na=False),
    ],
    [
        'test',
        'val',
    ],
    default='train'
)
df.value_counts('split')

split
train    22741
val       3184
test      2996
dtype: int64

In [11]:
split = df[['name', 'split']]

split.to_csv(Path(f'{output_dir}/split.tsv'), sep='\t')
print(Path(f'{output_dir}/split.tsv'))
split

../../../data/neims/gecko_EIMS_spectra/mist_inputs/msg/split.tsv


,name,split
0,msg-train_0,train
1,msg-train_1,train
2,msg-train_2,train
3,msg-train_3,train
4,msg-train_4,train
...,...,...
28919,msg-val_3179,val
28920,msg-val_3180,val
28921,msg-val_3181,val
28922,msg-val_3182,val


# Quinones

In [ ]:
path_to_data = f'../../../data/neims/gecko_EIMS_spectra'
assert Path(path_to_data).exists()
output_dir = f'{path_to_data}/mist_inputs/msg'
os.makedirs(output_dir, exist_ok = True)
fname = f"{path_to_data}/df_neims_msg_3_9_22.pkl"
with open(fname, 'rb') as file:
    df = pkl.load(file)
df